In [5]:
import os, pandas as pd
from sklearn.model_selection import train_test_split

IN_CSV = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/master_metadata_qc_keep30.csv"
OUTDIR = os.path.join(os.path.dirname(IN_CSV), "splits")
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(IN_CSV)
df["label"]  = df["label"].astype(str).str.lower().str.strip()
df["source"] = df["source"].astype(str).str.strip()
df["strata"] = df["label"] + "_" + df["source"]

seed = 42

# --- First split: stratify by label×source (bigger pool => usually safe)
train, temp = train_test_split(
    df, test_size=0.30, stratify=df["strata"], random_state=seed
)

# --- Second split: prefer label×source, but fallback to label-only if any stratum has <2
use_label_source = "strata" in temp and temp["strata"].value_counts().min() >= 2
strat2 = temp["strata"] if use_label_source else temp["label"]

val, test = train_test_split(
    temp, test_size=(1.0/3.0), stratify=strat2, random_state=seed
)

train.to_csv(os.path.join(OUTDIR, "train.csv"), index=False)
val.to_csv(  os.path.join(OUTDIR, "val.csv"),   index=False)
test.to_csv( os.path.join(OUTDIR, "test.csv"),  index=False)

def show(name, d):
    print(f"\n{name} n={len(d)}")
    print("by label:\n", d["label"].value_counts())
    print("by source:\n", d["source"].value_counts())

print("✅ Saved splits to:", OUTDIR)
print("Second split stratified by:", "label×source" if use_label_source else "label only")
show("TRAIN", train)
show("VAL",   val)
show("TEST",  test)


✅ Saved splits to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/splits
Second split stratified by: label only

TRAIN n=2506
by label:
 label
healthy    2485
cancer       21
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     1913
Cough_Audio_COUGHVID     570
kaggle_cancer_raw         21
kaggle_normal_raw          2
Name: count, dtype: int64

VAL n=716
by label:
 label
healthy    710
cancer       6
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     543
Cough_Audio_COUGHVID    167
kaggle_cancer_raw         6
Name: count, dtype: int64

TEST n=359
by label:
 label
healthy    356
cancer       3
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     277
Cough_Audio_COUGHVID     78
kaggle_cancer_raw         3
kaggle_normal_raw         1
Name: count, dtype: int64


In [7]:
# STEP 1: Patch split CSV paths to main folders (no symlinks)
import os, pandas as pd

SPLITS_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/splits"
train_csv, val_csv, test_csv = f"{SPLITS_DIR}/train.csv", f"{SPLITS_DIR}/val.csv", f"{SPLITS_DIR}/test.csv"

# Old vs new root segments (adjust only if needed)
OLD_ROOT = "/Project_PreProess_harmonize/Project"
NEW_ROOT = "/Project/Project"

def fix_df_paths(df):
    def fix(p):
        if isinstance(p, str) and os.path.exists(p):
            return p
        if isinstance(p, str) and OLD_ROOT in p:
            cand = p.replace(OLD_ROOT, NEW_ROOT)
            if os.path.exists(cand):
                return cand
        return p  # leave as-is if no fix found
    df["filepath"] = df["filepath"].apply(fix)

    # Report and drop any rows that still don't exist
    before = len(df)
    mask_ok = df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))
    dropped = (~mask_ok).sum()
    df = df[mask_ok].reset_index(drop=True)

    return df, before, dropped

for path in [train_csv, val_csv, test_csv]:
    df = pd.read_csv(path)
    df_fixed, before, dropped = fix_df_paths(df)
    df_fixed.to_csv(path, index=False)
    print(f"✅ Patched: {os.path.basename(path)} | kept {len(df_fixed)}/{before} | dropped missing: {dropped}")
    if "label" in df_fixed.columns:
        print("   by label:", df_fixed["label"].value_counts().to_dict())


✅ Patched: train.csv | kept 0/2506 | dropped missing: 2506
   by label: {}
✅ Patched: val.csv | kept 0/716 | dropped missing: 716
   by label: {}
✅ Patched: test.csv | kept 0/359 | dropped missing: 359
   by label: {}


In [13]:
import os, glob, pandas as pd

# === your four confirmed folders ===
COSWARA = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_Coswera/healthy"
COUGHVID= "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_COUGHVID/healthy2"
CANCER  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/cancer"
KAGGLE  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle"

# === where to save the new master metadata ===
MASTER_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata"
os.makedirs(MASTER_DIR, exist_ok=True)
OUT_CSV = os.path.join(MASTER_DIR, "master_metadata_realpaths.csv")

def scan_dir(root, label, source):
    exts = (".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp")
    rows = []
    if not os.path.isdir(root):
        print(f"❌ missing dir: {root}")
        return pd.DataFrame(rows)
    for dp,_,files in os.walk(root):
        for fn in files:
            if os.path.splitext(fn)[1].lower() in exts:
                fp = os.path.join(dp, fn)
                if os.path.exists(fp):
                    rows.append({"filepath": fp, "label": label, "source": source})
    return pd.DataFrame(rows)

# build the master from real paths (labels unified already)
df_cos  = scan_dir(COSWARA, "healthy", "Cough_Audio_Coswera")
df_cvv  = scan_dir(COUGHVID,"healthy", "Cough_Audio_COUGHVID")
df_can  = scan_dir(CANCER,  "cancer",  "kaggle_cancer_raw")
df_kag  = scan_dir(KAGGLE,  "healthy", "kaggle_normal_raw")

df = pd.concat([df_cos, df_cvv, df_can, df_kag], ignore_index=True)
df = df.drop_duplicates(subset=["filepath"]).reset_index(drop=True)

df.to_csv(OUT_CSV, index=False)

print(f"✅ saved: {OUT_CSV}")
print("counts by label:\n", df["label"].value_counts())
print("\ncounts by source:\n", df["source"].value_counts())
print("\nTOTAL files:", len(df))


✅ saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/master_metadata_realpaths.csv
counts by label:
 label
healthy    3670
cancer       30
Name: count, dtype: int64

counts by source:
 source
Cough_Audio_Coswera     2809
Cough_Audio_COUGHVID     837
kaggle_cancer_raw         30
kaggle_normal_raw         24
Name: count, dtype: int64

TOTAL files: 3700


In [14]:
# STEP 2: Recreate splits from the real-paths master (no symlinks)
import os, pandas as pd
from sklearn.model_selection import train_test_split

MASTER_CSV = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/master_metadata_realpaths.csv"
SPLITS_DIR = os.path.join(os.path.dirname(MASTER_CSV), "splits")
os.makedirs(SPLITS_DIR, exist_ok=True)

df = pd.read_csv(MASTER_CSV)

# 0) Keep only rows whose files exist (should be all, but we enforce)
exists = df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))
df = df[exists].reset_index(drop=True)

# 1) Normalise text + strata
df["label"]  = df["label"].astype(str).str.lower().str.strip()
df["source"] = df["source"].astype(str).str.strip()
df["strata"] = df["label"] + "_" + df["source"]

# 2) First split: stratify by label×source
seed = 42
train, temp = train_test_split(
    df, test_size=0.30, stratify=df["strata"], random_state=seed
)

# 3) Second split: prefer label×source; fallback to label-only if any stratum has < 2 in temp
use_label_source = temp["strata"].value_counts().min() >= 2
strat2 = temp["strata"] if use_label_source else temp["label"]

val, test = train_test_split(
    temp, test_size=(1.0/3.0), stratify=strat2, random_state=seed
)

# 4) Save
train_out = os.path.join(SPLITS_DIR, "train.csv")
val_out   = os.path.join(SPLITS_DIR, "val.csv")
test_out  = os.path.join(SPLITS_DIR, "test.csv")

train.to_csv(train_out, index=False)
val.to_csv(val_out, index=False)
test.to_csv(test_out, index=False)

def show(name, d):
    print(f"\n{name} n={len(d)}")
    print("by label:\n", d["label"].value_counts())
    print("by source:\n", d["source"].value_counts())

print("✅ Splits saved to:", SPLITS_DIR)
print("Second split stratified by:", "label×source" if use_label_source else "label only")
show("TRAIN", train)
show("VAL",   val)
show("TEST",  test)

# Quick existence sanity (should all be True)
print("\nExistence check (first 3 paths each):")
for name, d in [("train", train), ("val", val), ("test", test)]:
    ok = d["filepath"].apply(os.path.exists).all()
    print(f"- {name}: all_exist={ok}")
    for p in d["filepath"].head(3):
        print("   ", p)


✅ Splits saved to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/splits
Second split stratified by: label×source

TRAIN n=2590
by label:
 label
healthy    2569
cancer       21
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     1966
Cough_Audio_COUGHVID     586
kaggle_cancer_raw         21
kaggle_normal_raw         17
Name: count, dtype: int64

VAL n=740
by label:
 label
healthy    734
cancer       6
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     562
Cough_Audio_COUGHVID    167
kaggle_cancer_raw         6
kaggle_normal_raw         5
Name: count, dtype: int64

TEST n=370
by label:
 label
healthy    367
cancer       3
Name: count, dtype: int64
by source:
 source
Cough_Audio_Coswera     281
Cough_Audio_COUGHVID     84
kaggle_cancer_raw         3
kaggle_normal_raw         2
Name: count, dtype: int64

Existence check (first 3 paths each):
- train: all_exist=True

In [26]:
# REBUILD DATALOADERS: single-process (no multiprocessing / no shared memory)
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import torchvision.transforms as T
import torch.nn as nn
import random

SPLITS_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/splits"
TRAIN_CSV = f"{SPLITS_DIR}/train.csv"
VAL_CSV   = f"{SPLITS_DIR}/val.csv"
TEST_CSV  = f"{SPLITS_DIR}/test.csv"

LABEL2ID = {"healthy":0, "cancer":1}

# --- light class-conditional aug (same as before) ---
class SpecAugment2D(torch.nn.Module):
    def __init__(self, T_mask=30, F_mask=20, num_T=2, num_F=2, p=0.8):
        super().__init__(); self.T_mask=T_mask; self.F_mask=F_mask; self.num_T=num_T; self.num_F=num_F; self.p=p
    def forward(self, x):
        if not self.training or random.random()>self.p: return x
        c,H,W = x.shape
        for _ in range(self.num_T):
            w = random.randint(0, self.T_mask)
            if 0<w<W:
                t0 = random.randint(0, W-w); x[:, :, t0:t0+w] = 0.0
        for _ in range(self.num_F):
            h = random.randint(0, self.F_mask)
            if 0<h<H:
                f0 = random.randint(0, H-h); x[:, f0:f0+h, :] = 0.0
        return x

class RandomTimeShift(torch.nn.Module):
    def __init__(self, max_frac=0.08, p=0.6):
        super().__init__(); self.max_frac=max_frac; self.p=p
    def forward(self, x):
        if not self.training or random.random()>self.p: return x
        _,H,W = x.shape
        max_pix = max(1, int(self.max_frac*W))
        s = random.randint(-max_pix, max_pix)
        if s==0: return x
        pad = torch.zeros((1,H,abs(s)), dtype=x.dtype, device=x.device)
        return torch.cat([pad, x[:,:,:-s]], dim=2) if s>0 else torch.cat([x[:,:, -s:], pad], dim=2)

class AddGaussianNoise(torch.nn.Module):
    def __init__(self, sigma_min=0.005, sigma_max=0.02, p=0.5):
        super().__init__(); self.sigma_min=sigma_min; self.sigma_max=sigma_max; self.p=p
    def forward(self, x):
        if not self.training or random.random()>self.p: return x
        sigma = random.uniform(self.sigma_min, self.sigma_max)
        return (x + torch.randn_like(x)*sigma).clamp(0.0, 1.0)

def make_train_tf(minority=False):
    spec  = SpecAugment2D(T_mask=40 if minority else 30, F_mask=24 if minority else 20,
                          num_T=2, num_F=2, p=0.9 if minority else 0.7)
    shift = RandomTimeShift(max_frac=0.08 if minority else 0.05, p=0.7 if minority else 0.5)
    noise = AddGaussianNoise(0.005, 0.02, p=0.6 if minority else 0.3)
    return T.Compose([
        T.Lambda(lambda p: p.convert("L")),
        T.ToTensor(),
        spec, shift, noise,
        T.Lambda(lambda t: t.repeat(3,1,1)),
        T.RandomErasing(p=0.25 if minority else 0.10, scale=(0.01,0.05), ratio=(0.3,3.3), value=0.0),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])

eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class CSVImageDataset(Dataset):
    def __init__(self, csv_path, tf_healthy, tf_cancer):
        df = pd.read_csv(csv_path)
        df = df[df["filepath"].apply(lambda p: isinstance(p,str) and os.path.exists(p))].reset_index(drop=True)
        self.df = df; self.tf_h=tf_healthy; self.tf_c=tf_cancer
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = LABEL2ID[str(r["label"]).lower().strip()]
        x = Image.open(r["filepath"])
        tf = self.tf_c if y==1 else self.tf_h
        return tf(x), torch.tensor(y, dtype=torch.long)

# datasets
train_ds = CSVImageDataset(TRAIN_CSV, tf_healthy=make_train_tf(False), tf_cancer=make_train_tf(True))
val_ds   = CSVImageDataset(VAL_CSV,   tf_healthy=eval_tf,               tf_cancer=eval_tf)
test_ds  = CSVImageDataset(TEST_CSV,  tf_healthy=eval_tf,               tf_cancer=eval_tf)

# oversampling weights
y_train = train_ds.df["label"].map(LABEL2ID).values
counts = np.bincount(y_train, minlength=2)
class_sampling_w = (counts.sum() / (2.0*np.maximum(counts,1))).astype(np.float32)
per_sample_w = class_sampling_w[y_train]

sampler = WeightedRandomSampler(per_sample_w.tolist(), num_samples=len(per_sample_w), replacement=True)

# single-process loaders (no shared memory / workers)
BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=0, pin_memory=False, persistent_workers=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False, persistent_workers=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,   num_workers=0, pin_memory=False, persistent_workers=False)

# sanity
print("num_workers -> train/val/test:", train_loader.num_workers, val_loader.num_workers, test_loader.num_workers)
xb, yb = next(iter(train_loader))
print("Train batch:", tuple(xb.shape), "| first labels:", yb[:12].tolist())
print("Train counts [healthy,cancer]:", counts.tolist())
print("Sampler class weights:", class_sampling_w.tolist())
print("Sizes -> train/val/test:", len(train_ds), len(val_ds), len(test_ds))


num_workers -> train/val/test: 0 0 0
Train batch: (32, 3, 224, 224) | first labels: [0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1]
Train counts [healthy,cancer]: [2569, 21]
Sampler class weights: [0.5040872097015381, 61.66666793823242]
Sizes -> train/val/test: 2590 740 370


In [27]:
import os, numpy as np, torch
from torch import nn
from torch.optim import AdamW
from torchvision import models
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_recall_fscore_support

# reuse: train_loader, val_loader, test_loader, train_ds
assert 'train_loader' in globals() and 'val_loader' in globals() and 'test_loader' in globals()

# ---- device & reproducibility
torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# ---- model
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# ---- class-weighted loss from TRAIN
LABEL2ID = {"healthy":0, "cancer":1}
y_train = train_ds.df["label"].map(LABEL2ID).values
counts = np.bincount(y_train, minlength=2)
class_w = torch.tensor([counts.sum()/(2*max(1,counts[0])),
                        counts.sum()/(2*max(1,counts[1]))],
                       dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)

# ---- optim & scheduler (no 'verbose' arg)
opt = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

def run_epoch(loader, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        losses.append(loss.item())
        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(probs.tolist())
    ys, ps = np.array(ys), np.array(ps)
    yhat = (ps >= 0.5).astype(int)
    acc = accuracy_score(ys, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(ys, yhat, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(ys, ps)
    except ValueError:
        auc = float('nan')
    return float(np.mean(losses)), acc, prec, rec, f1, auc

# ---- training loop with early stop on val AUC + safe checkpointing
EPOCHS, PATIENCE = 15, 4
best_auc, best_state, best_epoch, no_improve = -1.0, None, -1, 0

ckpt_dir = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
save_path = os.path.join(ckpt_dir, "resnet18_best.pt")

for ep in range(1, EPOCHS+1):
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    sched.step(vl[-1])
    print(f"Epoch {ep:02d} | train loss {tr[0]:.4f} f1 {tr[4]:.3f} auc {tr[5]:.3f} | "
          f"val loss {vl[0]:.4f} f1 {vl[4]:.3f} auc {vl[5]:.3f}")

    if vl[-1] > best_auc:
        best_auc, best_epoch = vl[-1], ep
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        # safe save (handles low disk)
        try:
            torch.save(best_state, save_path)
        except OSError:
            # fallback: save half precision, smaller file
            save_path = os.path.join(ckpt_dir, "resnet18_best_fp16.pt")
            best_state_fp16 = {k: (v.half() if v.dtype.is_floating_point else v) for k, v in best_state.items()}
            torch.save(best_state_fp16, save_path)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val AUC at {best_epoch}: {best_auc:.3f})")
            break

print(f"\nBest epoch: {best_epoch} | Best val AUC: {best_auc:.3f}")
print("Saved:", save_path)

# ---- test using best
if best_state is not None:
    model.load_state_dict(best_state, strict=False)
test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = run_epoch(test_loader, train=False)
print("\nTEST METRICS")
print(f"loss {test_loss:.4f} | acc {test_acc:.3f} | prec {test_prec:.3f} | rec {test_rec:.3f} | f1 {test_f1:.3f} | auc {test_auc:.3f}")


Device: cuda
Epoch 01 | train loss 0.0090 f1 0.988 auc 0.999 | val loss 0.0184 f1 0.800 auc 0.999
Epoch 02 | train loss 0.0003 f1 0.998 auc 1.000 | val loss 0.0189 f1 0.706 auc 0.999
Epoch 03 | train loss 0.0005 f1 0.996 auc 0.999 | val loss 0.0081 f1 0.800 auc 0.999
Epoch 04 | train loss 0.0000 f1 1.000 auc 1.000 | val loss 0.0093 f1 0.800 auc 1.000
Epoch 05 | train loss 0.0002 f1 0.998 auc 1.000 | val loss 0.0100 f1 0.923 auc 1.000
Epoch 06 | train loss 0.0000 f1 1.000 auc 1.000 | val loss 0.0021 f1 0.923 auc 1.000
Epoch 07 | train loss 0.0000 f1 1.000 auc 1.000 | val loss 0.0024 f1 0.923 auc 1.000
Epoch 08 | train loss 0.0000 f1 1.000 auc 1.000 | val loss 0.0044 f1 0.923 auc 1.000
Epoch 09 | train loss 0.0000 f1 1.000 auc 1.000 | val loss 0.0528 f1 0.833 auc 1.000
Early stopping at epoch 9 (best val AUC at 5: 1.000)

Best epoch: 5 | Best val AUC: 1.000
Saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Mas

In [28]:
# STEP 5: Export predictions for VAL/TEST + basic metrics at threshold 0.5
import os, numpy as np, pandas as pd, torch
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

assert 'model' in globals()
device = next(model.parameters()).device

# Ensure we're using the best weights already (you printed a path earlier)
# If needed, uncomment and set the correct path:
# ckpt = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/checkpoints/resnet18_best.pt"
# state = torch.load(ckpt, map_location=device)
# model.load_state_dict(state, strict=False)

LABEL2ID = {"healthy":0, "cancer":1}
ID2LABEL = {0:"healthy", 1:"cancer"}

def collect_preds(loader, dataset):
    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            pr = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            probs.extend(pr); ys.extend(yb.numpy())
    df = dataset.df.copy()  # order matches because shuffle=False
    df["y_true"] = ys
    df["prob_cancer"] = probs
    df["y_pred_0p5"] = (df["prob_cancer"] >= 0.5).astype(int)
    return df

# Collect
val_df  = collect_preds(val_loader,  val_ds)
test_df = collect_preds(test_loader, test_ds)

# Metrics helpers
def summarize(df, split_name):
    y = df["y_true"].values
    p = df["prob_cancer"].values
    yhat = df["y_pred_0p5"].values
    auc = roc_auc_score(y, p)
    cm  = confusion_matrix(y, yhat, labels=[0,1])
    print(f"\n[{split_name}] AUC={auc:.3f}")
    print("Confusion matrix (rows=true [healthy,cancer], cols=pred):\n", cm)
    print("Classification report:")
    print(classification_report(y, yhat, target_names=["healthy","cancer"], zero_division=0))
    # per-source snapshot
    by_src = df.groupby(["source","label"]).agg(
        n=("y_true","size"),
        pos=("y_true", lambda a: int((np.array(a)==1).sum())),
        tp=("y_pred_0p5", lambda a: int(((np.array(a)==1) & (df.loc[a.index,'y_true']==1)).sum())),
        fp=("y_pred_0p5", lambda a: int(((np.array(a)==1) & (df.loc[a.index,'y_true']==0)).sum())),
    ).reset_index()
    print("Per-source counts (first 8 rows):")
    print(by_src.head(8).to_string(index=False))

summarize(val_df,  "VAL @0.5")
summarize(test_df, "TEST @0.5")

# Save predictions
out_dir = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/predictions"
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, "resnet18_val_test_predictions.csv")
pd.concat([val_df.assign(split="val"), test_df.assign(split="test")], ignore_index=True).to_csv(out_csv, index=False)
print(f"\n✅ Saved predictions to: {out_csv}")



[VAL @0.5] AUC=1.000
Confusion matrix (rows=true [healthy,cancer], cols=pred):
 [[733   1]
 [  0   6]]
Classification report:
              precision    recall  f1-score   support

     healthy       1.00      1.00      1.00       734
      cancer       0.86      1.00      0.92         6

    accuracy                           1.00       740
   macro avg       0.93      1.00      0.96       740
weighted avg       1.00      1.00      1.00       740

Per-source counts (first 8 rows):
              source   label   n  pos  tp  fp
Cough_Audio_COUGHVID healthy 167    0   0   0
 Cough_Audio_Coswera healthy 562    0   0   0
   kaggle_cancer_raw  cancer   6    6   6   0
   kaggle_normal_raw healthy   5    0   0   1

[TEST @0.5] AUC=1.000
Confusion matrix (rows=true [healthy,cancer], cols=pred):
 [[366   1]
 [  0   3]]
Classification report:
              precision    recall  f1-score   support

     healthy       1.00      1.00      1.00       367
      cancer       0.75      1.00      0.86  

In [29]:
# Tune threshold τ on VAL, apply to TEST. Uses val_df/test_df already created.
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, roc_auc_score

def metrics_at(y_true, p_cancer, tau):
    yhat = (p_cancer >= tau).astype(int)
    acc = accuracy_score(y_true, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, yhat, average='binary', zero_division=0)
    cm = confusion_matrix(y_true, yhat, labels=[0,1])
    return acc, prec, rec, f1, cm

yv, pv = val_df["y_true"].values,  val_df["prob_cancer"].values
yt, pt = test_df["y_true"].values, test_df["prob_cancer"].values

# 1) τ_F1: threshold that maximises F1 on VAL
taus = np.linspace(0, 1, 501)
best = (-1, 0.5, None)  # (F1, tau, (acc,prec,rec,f1,cm))
for tau in taus:
    acc, prec, rec, f1, cm = metrics_at(yv, pv, tau)
    if f1 > best[0]:
        best = (f1, tau, (acc,prec,rec,f1,cm))
tau_f1 = float(best[1])

# 2) τ_P: highest threshold achieving precision >= 0.98 on VAL (guard FP), choose max recall under that constraint
cands = []
for tau in taus:
    acc, prec, rec, f1, cm = metrics_at(yv, pv, tau)
    if prec >= 0.98:
        cands.append((rec, tau, (acc,prec,rec,f1,cm)))
if cands:
    cands.sort(reverse=True)                       # highest recall first
    tau_p = float(cands[0][1])
    val_p = cands[0][2]
else:
    tau_p, val_p = None, None

def show(split_name, y, p, tau, label):
    acc, prec, rec, f1, cm = metrics_at(y, p, tau)
    print(f"[{split_name} @ {label} τ={tau:.3f}]  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}")
    print("  CM (rows=true [healthy,cancer], cols=pred):")
    print(cm)

# Print choices on VAL
print(f"VAL: best-F1 threshold τ_F1 = {tau_f1:.3f}")
acc,prec,rec,f1,cm = metrics_at(yv, pv, tau_f1)
print(f"  -> acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}")
print("  CM:\n", cm)

if tau_p is not None:
    print(f"\nVAL: precision-target threshold τ_P (prec≥0.98) = {tau_p:.3f}")
    a,p,r,f,cm2 = val_p
    print(f"  -> acc={a:.3f}  prec={p:.3f}  rec={r:.3f}  f1={f:.3f}")
    print("  CM:\n", cm2)
else:
    print("\nVAL: No τ achieves precision ≥ 0.98; keep τ_F1.")

# Apply both thresholds to TEST
print("\n--- Apply to TEST ---")
show("TEST", yt, pt, tau_f1, "τ_F1")
if tau_p is not None:
    show("TEST", yt, pt, tau_p,  "τ_P ")


VAL: best-F1 threshold τ_F1 = 0.678
  -> acc=1.000  prec=1.000  rec=1.000  f1=1.000
  CM:
 [[734   0]
 [  0   6]]

VAL: precision-target threshold τ_P (prec≥0.98) = 0.762
  -> acc=1.000  prec=1.000  rec=1.000  f1=1.000
  CM:
 [[734   0]
 [  0   6]]

--- Apply to TEST ---
[TEST @ τ_F1 τ=0.678]  acc=1.000  prec=1.000  rec=1.000  f1=1.000
  CM (rows=true [healthy,cancer], cols=pred):
[[367   0]
 [  0   3]]
[TEST @ τ_P  τ=0.762]  acc=1.000  prec=1.000  rec=1.000  f1=1.000
  CM (rows=true [healthy,cancer], cols=pred):
[[367   0]
 [  0   3]]


In [30]:
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score

# we already computed these above:
# val_df, test_df with columns: [filepath, label, source, y_true, prob_cancer, y_pred_0p5]
# tau_f1, tau_p (tau_p may be None)

# 1) choose operating threshold
tau_star = float(tau_p) if 'tau_p' in globals() and tau_p is not None else float(tau_f1)
print(f"Operating threshold τ* = {tau_star:.3f}")

def metrics_at(df, tau):
    y = df["y_true"].values
    p = df["prob_cancer"].values
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    cm = confusion_matrix(y, yhat, labels=[0,1])
    auc = roc_auc_score(y, p)
    return {"acc":acc, "prec":prec, "rec":rec, "f1":f1, "auc":auc, "cm":cm}

def per_source_table(df, tau):
    rows = []
    for (src, lab), g in df.groupby(["source","label"]):
        y = g["y_true"].values
        p = g["prob_cancer"].values
        yhat = (p >= tau).astype(int)
        acc = accuracy_score(y, yhat)
        prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
        cm = confusion_matrix(y, yhat, labels=[0,1])
        rows.append({
            "source": src, "label": lab, "n": len(g),
            "acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        })
    return pd.DataFrame(rows).sort_values(["source","label","n"], ascending=[True, True, False])

# 2) overall metrics
val_over = metrics_at(val_df,  tau_star)
test_over= metrics_at(test_df, tau_star)

print("\n[VAL @ τ*]  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}".format(**val_over))
print("CM:\n", val_over["cm"])
print("\n[TEST @ τ*] acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}".format(**test_over))
print("CM:\n", test_over["cm"])

# 3) per-source tables
val_src  = per_source_table(val_df,  tau_star)
test_src = per_source_table(test_df, tau_star)
print("\nVAL per-source (head):\n", val_src.head(8).to_string(index=False))
print("\nTEST per-source (head):\n", test_src.head(8).to_string(index=False))

# 4) Kaggle-out sanity (exclude kaggle_* sources)
def exclude_kaggle(df):
    m = ~df["source"].str.contains("kaggle_", na=False)
    return df[m].reset_index(drop=True)

val_nok = exclude_kaggle(val_df)
test_nok= exclude_kaggle(test_df)

if len(val_nok) and len(test_nok):
    val_over_nok = metrics_at(val_nok, tau_star)
    test_over_nok= metrics_at(test_nok, tau_star)
    print("\n[VAL no-Kaggle @ τ*]  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}".format(**val_over_nok))
    print("[TEST no-Kaggle @ τ*] acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}".format(**test_over_nok))
else:
    print("\n(no-Kaggle subset too small; skipping)")

# 5) save for the report
rep_dir = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports"
os.makedirs(rep_dir, exist_ok=True)
val_src.to_csv(os.path.join(rep_dir, "val_per_source_tau_star.csv"), index=False)
test_src.to_csv(os.path.join(rep_dir, "test_per_source_tau_star.csv"), index=False)

with open(os.path.join(rep_dir, "operating_threshold.txt"), "w") as f:
    f.write(f"tau_star={tau_star:.6f}\nchosen='precision>=0.98 on VAL' if available else 'best-F1 on VAL'\n")

print(f"\n✅ Saved per-source tables + τ* under: {rep_dir}")


Operating threshold τ* = 0.762

[VAL @ τ*]  acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
CM:
 [[734   0]
 [  0   6]]

[TEST @ τ*] acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
CM:
 [[367   0]
 [  0   3]]

VAL per-source (head):
               source   label   n  acc  prec  rec  f1  tn  fp  fn  tp
Cough_Audio_COUGHVID healthy 167  1.0   0.0  0.0 0.0 167   0   0   0
 Cough_Audio_Coswera healthy 562  1.0   0.0  0.0 0.0 562   0   0   0
   kaggle_cancer_raw  cancer   6  1.0   1.0  1.0 1.0   0   0   0   6
   kaggle_normal_raw healthy   5  1.0   0.0  0.0 0.0   5   0   0   0

TEST per-source (head):
               source   label   n  acc  prec  rec  f1  tn  fp  fn  tp
Cough_Audio_COUGHVID healthy  84  1.0   0.0  0.0 0.0  84   0   0   0
 Cough_Audio_Coswera healthy 281  1.0   0.0  0.0 0.0 281   0   0   0
   kaggle_cancer_raw  cancer   3  1.0   1.0  1.0 1.0   0   0   0   3
   kaggle_normal_raw healthy   2  1.0   0.0  0.0 0.0   2   0   0   0

[VAL no-Kaggle @ τ*]  acc=1.000  prec

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [31]:
# Kaggle-only evaluation from the already computed val_df/test_df and tau_star
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score

def eval_subset(df, name, tau):
    sub = df[df["source"].str.contains("kaggle_", na=False)].copy()
    y = sub["y_true"].values
    p = sub["prob_cancer"].values
    yhat = (p >= tau).astype(int)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(y, p)
    except ValueError:
        auc = float('nan')
    cm = confusion_matrix(y, yhat, labels=[0,1])
    print(f"\n[{name} Kaggle-only @ τ*={tau_star:.3f}]  acc={acc:.3f}  prec={prec:.3f}  rec={rec:.3f}  f1={f1:.3f}  auc={auc:.3f}")
    print("  CM (rows=true [healthy,cancer], cols=pred):\n", cm)
    print("  Breakdown:\n", sub["label"].value_counts().to_string())
    return sub

val_k = eval_subset(val_df,  "VAL",  tau_star)
test_k= eval_subset(test_df, "TEST", tau_star)



[VAL Kaggle-only @ τ*=0.762]  acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
  CM (rows=true [healthy,cancer], cols=pred):
 [[5 0]
 [0 6]]
  Breakdown:
 label
cancer     6
healthy    5

[TEST Kaggle-only @ τ*=0.762]  acc=1.000  prec=1.000  rec=1.000  f1=1.000  auc=1.000
  CM (rows=true [healthy,cancer], cols=pred):
 [[2 0]
 [0 3]]
  Breakdown:
 label
cancer     3
healthy    2


In [32]:
# Freeze artifacts and add a tiny inference utility
import os, json, random, torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import roc_auc_score

# ---- paths/artifacts from earlier steps ----
ckpt_dir  = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/checkpoints"
save_path = os.path.join(ckpt_dir, "resnet18_best.pt")          # already saved
rep_dir   = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports"
os.makedirs(rep_dir, exist_ok=True)

# Operating threshold τ* we selected
tau_star = 0.762  # from your previous output

# Label map
label2id = {"healthy": 0, "cancer": 1}
id2label = {v:k for k,v in label2id.items()}

# ---- save minimal metadata for reproducibility ----
with open(os.path.join(rep_dir, "label_map.json"), "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

with open(os.path.join(rep_dir, "operating_threshold.txt"), "w") as f:
    f.write(f"{tau_star:.6f}\n")

print("✅ Saved label_map.json and operating_threshold.txt")

# ---- define the exact eval transform used (grayscale->3ch + ImageNet norm) ----
eval_tf = T.Compose([
    T.Lambda(lambda p: p.convert("L")),
    T.ToTensor(),
    T.Lambda(lambda t: t.repeat(3,1,1)),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ---- load model with best weights ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(model.fc.in_features, 2)
state = torch.load(save_path, map_location=device)
model.load_state_dict(state, strict=False)
model.eval().to(device)
print("✅ Loaded model:", save_path, "| device:", device)

# ---- inference helper ----
@torch.inference_mode()
def predict_image(path, tau=0.5):
    img = Image.open(path)
    x = eval_tf(img).unsqueeze(0).to(device)   # [1,3,224,224]
    logits = model(x)
    p = torch.softmax(logits, dim=1)[0,1].item()
    yhat = int(p >= tau)
    return p, yhat

# ---- try 5 random TEST images for a sanity check ----
import pandas as pd
SPLITS_DIR = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/metadata/splits"
test_df = pd.read_csv(os.path.join(SPLITS_DIR, "test.csv")).sample(5, random_state=7).reset_index(drop=True)

print("\nSample predictions (τ* applied):")
for _, r in test_df.iterrows():
    p, yhat = predict_image(r["filepath"], tau=tau_star)
    print(f"- {os.path.basename(r['filepath'])[:40]:<40} "
          f"true={r['label']:<7}  p(cancer)={p:0.3f}  pred={id2label[yhat]}")


✅ Saved label_map.json and operating_threshold.txt
✅ Loaded model: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/checkpoints/resnet18_best.pt | device: cuda

Sample predictions (τ* applied):
- 2owO9U3KZ8TqQDrOx6UhOiA2OW43_cough-shall true=healthy  p(cancer)=0.000  pred=healthy
- 4e9u1T1conYlXIQp89SQRgMPScY2_cough-shall true=healthy  p(cancer)=0.000  pred=healthy
- 6VbC2mWfKiWVQPL1tHVmG5eCkua2_cough-shall true=healthy  p(cancer)=0.000  pred=healthy
- 9esRO6MV7jhuNNWR9i0j6gBrkXG2_cough-shall true=healthy  p(cancer)=0.000  pred=healthy
- 2j94Kg8TewQwjwbrRyto6seGsut1_cough-shall true=healthy  p(cancer)=0.000  pred=healthy


In [34]:
# Reload predictions from disk, then plot
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix

# 1) Load the predictions we saved earlier
PRED_CSV = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/predictions/resnet18_val_test_predictions.csv"
df_pred  = pd.read_csv(PRED_CSV)

# Sanity: show columns and split counts
print("Columns:", list(df_pred.columns))
print("Split counts:", df_pred["split"].value_counts().to_dict())

# Split back into VAL/TEST DataFrames the plotting code expects
val_df  = df_pred[df_pred["split"]=="val"].reset_index(drop=True).copy()
test_df = df_pred[df_pred["split"]=="test"].reset_index(drop=True).copy()

# 2) Prepare plotting helpers
rep_dir = "/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports"
os.makedirs(rep_dir, exist_ok=True)

tau_star = 0.762  # from threshold tuning

def kaggle_only(df):
    return df[df["source"].str.contains("kaggle_", na=False)].reset_index(drop=True)

def plot_roc(y, p, title, out_png):
    fpr, tpr, _ = roc_curve(y, p)
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    return out_png, roc_auc

def plot_pr(y, p, title, out_png):
    prec, rec, _ = precision_recall_curve(y, p)
    # (Area under PR curve using trapezoid rule—fine for reporting alongside ROC)
    ap = np.trapz(prec, rec)
    plt.figure()
    plt.plot(rec, prec, label=f"Area ≈ {ap:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(title)
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    return out_png, ap

def plot_cm(y, p, tau, title, out_png):
    yhat = (p >= tau).astype(int)
    cm = confusion_matrix(y, yhat, labels=[0,1])
    plt.figure()
    plt.imshow(cm, cmap="Blues")
    for (i,j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.xticks([0,1], ["Pred: healthy","Pred: cancer"], rotation=15)
    plt.yticks([0,1], ["True: healthy","True: cancer"])
    plt.title(title + f"\nτ*={tau:.3f}")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    return out_png, cm

# 3) Kaggle-only subsets and arrays
val_k = kaggle_only(val_df)
test_k= kaggle_only(test_df)
yv, pv = val_k["y_true"].values,  val_k["prob_cancer"].values
yt, pt = test_k["y_true"].values, test_k["prob_cancer"].values

# 4) Make and save figures
val_roc_png, val_auc = plot_roc(yv, pv, "ROC — Kaggle-only (VAL)",  os.path.join(rep_dir, "roc_kaggle_val.png"))
val_pr_png,  val_ap  = plot_pr (yv, pv, "PR  — Kaggle-only (VAL)",  os.path.join(rep_dir, "pr_kaggle_val.png"))
val_cm_png,  val_cm  = plot_cm (yv, pv, tau_star, "Confusion Matrix — Kaggle-only (VAL)",  os.path.join(rep_dir, "cm_kaggle_val_tauStar.png"))

test_roc_png, test_auc = plot_roc(yt, pt, "ROC — Kaggle-only (TEST)", os.path.join(rep_dir, "roc_kaggle_test.png"))
test_pr_png,  test_ap  = plot_pr (yt, pt, "PR  — Kaggle-only (TEST)", os.path.join(rep_dir, "pr_kaggle_test.png"))
test_cm_png,  test_cm  = plot_cm (yt, pt, tau_star, "Confusion Matrix — Kaggle-only (TEST)", os.path.join(rep_dir, "cm_kaggle_test_tauStar.png"))

print("\nSaved:")
for p in [val_roc_png, val_pr_png, val_cm_png, test_roc_png, test_pr_png, test_cm_png]:
    print(" -", p)
print(f"\nVAL AUC={val_auc:.3f} | TEST AUC={test_auc:.3f}")


Columns: ['filepath', 'label', 'source', 'strata', 'y_true', 'prob_cancer', 'y_pred_0p5', 'split']
Split counts: {'val': 740, 'test': 370}

Saved:
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports/roc_kaggle_val.png
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports/pr_kaggle_val.png
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports/cm_kaggle_val_tauStar.png
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports/roc_kaggle_test.png
 - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Final_Master/reports/pr_kaggle_test.png
 - /workspaces/cmp9137-advanced-machine-learning/CM